### **Rôle principal du routeur d'Audit :**
Ce fichier agit comme le registre de traçabilité (Audit Trail) de votre système d'aide à la décision clinique. Il expose une série de points d'accès (endpoints) d'API via FastAPI permettant d'enregistrer et de consulter les décisions prises par les médecins face aux alertes de sécurité générées par le moteur CDS.

### **Étapes et mécanismes implémentés :**

##### **Étape 1 : Enregistrement individuel d'une décision (POST /)**

- Ce point de terminaison permet à un praticien d'enregistrer son choix final face à une alerte spécifique (accepter l'alerte, l'ignorer, ou forcer la prescription / "OVERRIDE").

- Le système vérifie d'abord que l'alerte existe bien en base de données avant d'enregistrer l'action du médecin.

- Il crée une archive statique en capturant le type, la sévérité et le titre de l'alerte au moment précis de la décision.

##### **Étape 2 : Validation IA des forçages (Intégrée au POST /)**

- Si le médecin choisit de forcer la prescription ("OVERRIDE") et fournit une justification textuelle, le système déclenche un appel au service d'intelligence artificielle (validate_override_justification).

- L'IA évalue si la justification médicale est valide ("valid") ou s'il s'agit de texte inutile ("noise"), et ajoute un retour (feedback) directement dans l'entrée d'audit.

##### **Étape 3 : Traitement par lots (POST /bulk)**

- Plutôt que d'envoyer une requête par alerte, le front-end peut utiliser cette route pour valider toutes les décisions d'une ordonnance en une seule fois.

- Ce processus est tolérant aux erreurs : si une alerte n'est pas trouvée, le système l'ignore silencieusement et continue d'enregistrer le reste des décisions pour ne pas bloquer le flux de travail du médecin.

- Chaque décision "OVERRIDE" du lot subit sa propre validation sémantique par le LLM.

##### **Étape 4 : Historique clinique par ordonnance (GET /prescription/{prescription_id})**

- Cette route permet de récupérer l'historique complet de toutes les alertes et décisions liées à une ordonnance spécifique.

- Les résultats sont triés chronologiquement, du plus récent au plus ancien, afin de reconstruire le fil de pensée du praticien lors de la prescription.

##### **Étape 5 : Supervision du système (GET /recent)**

- Il s'agit d'une route de surveillance globale protégée, strictement réservée aux administrateurs (contrôle du rôle Role.ADMIN).

- Elle permet de récupérer un flux des 50 à 200 dernières décisions prises sur l'ensemble du système, offrant une vue d'ensemble sur le comportement de prescription et les alertes fréquemment ignorées.